# Baseload hedge for the retail book

Our retail book is about 1% of national demand. Customers pay a fixed tariff, we buy the energy
in the day-ahead market, so we are short spot price. The desk hedges with monthly baseload
forwards. This notebook estimates the minimum-variance hedge ratio, its effectiveness, and the
95% VaR of the hedged book from 24 monthly observations (2022–2023).

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

## Load data and build the book

In [2]:
df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"]).set_index("time")
SHARE = 0.01
TARIFF = 125.0                                   # EUR/MWh fixed customer price

df["load"] = df["consumption_mwh"] * SHARE       # MWh per hour
df["cost"] = df["price_eur_mwh"] * df["load"]    # EUR per hour
df["revenue"] = TARIFF * df["load"]
print(df.shape)
df[["price_eur_mwh", "load", "cost"]].describe().round(1)

(17520, 8)


,price_eur_mwh,load,cost
count,17520.0,17520.0,17520.0
mean,98.5,293.2,29777.0
std,36.9,42.0,13744.7
min,-19.9,180.9,-5789.8
25%,73.5,264.5,19715.1
50%,97.7,296.7,28324.2
75%,122.8,323.7,38712.6
max,419.6,408.2,143031.2


## Forward price and hedge instrument

The forward curve is flat at the long-run average price. The hedge instrument is a baseload
strip: a constant volume `V` MW in every hour, paying `(spot - F)` per MWh.

In [3]:
F = df["price_eur_mwh"].mean()
V = df["load"].mean()
df["payoff"] = (df["price_eur_mwh"] - F) * V
print(f"F = {F:.2f} EUR/MWh, V = {V:.1f} MW")

F = 98.52 EUR/MWh, V = 293.2 MW


## Minimum-variance hedge ratio

The optimal ratio is `cov(cost, payoff) / var(payoff)`. Effectiveness is the R² of the same
regression.

In [4]:
cov = np.cov(df["cost"], df["payoff"], ddof=0)[0, 1]
h = cov / df["payoff"].var()
r2 = np.corrcoef(df["cost"], df["payoff"])[0, 1] ** 2
resid = df["cost"] - h * df["payoff"]
print(f"hedge ratio h = {h:.3f}")
print(f"effectiveness  = {r2:.3f}")
print(f"residual sd    = {resid.std():.0f} EUR")

hedge ratio h = 1.223
effectiveness  = 0.927
residual sd    = 3715 EUR


In [5]:
by_year = pd.DataFrame({"year": df.index.year, "cost": df["cost"], "payoff": df["payoff"]})
by_year.groupby("year")[["cost", "payoff"]].apply(lambda g: g["cost"].cov(g["payoff"]) / g["payoff"].var()).round(3)

year
2022    1.256
2023    1.269
dtype: float64

## Monthly view

In [6]:
monthly = pd.DataFrame({
    "avg_price": df["price_eur_mwh"].resample("MS").mean(),
    "load": df["load"].resample("MS").sum(),
    "hours": df["load"].resample("MS").size(),
})
monthly["cost"] = monthly["avg_price"] * monthly["load"]
monthly["revenue"] = TARIFF * monthly["load"]
monthly["payoff"] = (monthly["avg_price"] - F) * V * monthly["hours"]
monthly.round(0).head(6)

,avg_price,load,hours,cost,revenue,payoff
time,,,,,,
2022-01-01 00:00:00+00:00,103.0,241699.0,744,25002808.0,30212370.0,1073484.0
2022-02-01 00:00:00+00:00,101.0,212808.0,672,21528959.0,26600981.0,520465.0
2022-03-01 00:00:00+00:00,98.0,232963.0,744,22860092.0,29120375.0,-86511.0
2022-04-01 00:00:00+00:00,81.0,207352.0,720,16882527.0,25918981.0,-3610246.0
2022-05-01 00:00:00+00:00,96.0,207608.0,744,19995826.0,25951037.0,-481807.0
2022-06-01 00:00:00+00:00,102.0,196385.0,720,20047256.0,24548185.0,750762.0


In [7]:
monthly["margin"] = monthly["revenue"] - monthly["cost"]
monthly["margin_hedged"] = monthly["margin"] - h * monthly["payoff"]
monthly[["margin", "margin_hedged"]].describe().round(0)

,margin,margin_hedged
count,24.0,24.0
mean,5538422.0,5538422.0
std,4661901.0,10650015.0
min,-3997219.0,-15437268.0
25%,1818834.0,-2946201.0
50%,5666813.0,5474001.0
75%,8985656.0,13104779.0
max,12933368.0,23139062.0


Seasonality of the book across the 24 months:

In [8]:
season = df.groupby(df.index.month).agg(avg_price=("price_eur_mwh", "mean"),
                                         load=("load", "sum"), cost=("cost", "sum"))
season["margin"] = TARIFF * season["load"] - season["cost"]
season.round(0)

,avg_price,load,cost,margin
time,,,,
1,111.0,482250.0,54541798.0,5739479.0
2,106.0,426290.0,46043593.0,7242675.0
3,100.0,461925.0,47165371.0,10575250.0
4,82.0,416360.0,35176908.0,16868062.0
5,84.0,415999.0,35912413.0,16087474.0
6,81.0,392341.0,32604905.0,16437760.0
7,90.0,404843.0,37406752.0,13198656.0
8,95.0,406185.0,39683584.0,11089595.0
9,98.0,397681.0,40236271.0,9473830.0


## Scenario VaR

Bootstrap 500 synthetic months by drawing 30 days with replacement from the sample, compute the
hedged margin in each, and read off the 95% VaR.

In [9]:
rng = np.random.default_rng(0)
days = [g for _, g in df.groupby(df.index.date)]
scen = []
for _ in range(500):
    pick = rng.integers(0, len(days), 30)
    m = pd.concat([days[i] for i in pick])
    cost = m["cost"].sum()
    rev = m["revenue"].sum()
    payoff = (m["price_eur_mwh"].mean() - F) * V * len(m)
    scen.append({"margin": rev - cost, "margin_hedged": rev - cost - h * payoff})
scen = pd.DataFrame(scen)
scen.describe().round(0)

,margin,margin_hedged
count,500.0,500.0
mean,4980482.0,5026958.0
std,941479.0,2142176.0
min,2580073.0,-318790.0
25%,4357002.0,3639027.0
50%,5033172.0,5091876.0
75%,5578561.0,6368957.0
max,7555543.0,11331281.0


In [10]:
var95 = np.percentile(scen["margin_hedged"], 95)
var95_unhedged = np.percentile(scen["margin"], 95)
print(f"95% VaR hedged:   {var95:,.0f} EUR")
print(f"95% VaR unhedged: {var95_unhedged:,.0f} EUR")

95% VaR hedged:   8,369,232 EUR
95% VaR unhedged: 6,442,989 EUR


## Results

In [11]:
pd.Series({
    "hedge ratio (x baseload)": round(h, 2),
    "hedge effectiveness": round(r2, 3),
    "residual risk (EUR)": round(resid.std()),
    "95% VaR hedged (EUR)": round(var95),
    "95% VaR unhedged (EUR)": round(var95_unhedged),
})

hedge ratio (x baseload)          1.220
hedge effectiveness               0.927
residual risk (EUR)            3715.000
95% VaR hedged (EUR)        8369232.000
95% VaR unhedged (EUR)      6442989.000
dtype: float64

Hedging 1.2x the average load with baseload forwards removes 93% of the cost variance. The
residual risk is around a thousand euros, i.e. negligible, and the hedged VaR is positive: even
in the 95% worst case the hedged book makes money. Recommendation: move the hedge ratio from 1.0
to 1.2.